In [1]:
import os
import glob
import numpy as np
import torch
import cv2
import rasterio
import pandas as pd
from segment_anything import sam_model_registry, SamPredictor
from PIL import Image
from skimage.transform import resize

# ======== Initialize SAM Model ========
sam_checkpoint = "sam_vit_h_4b8939.pth"
model_type = "vit_h"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint).to(device)
predictor = SamPredictor(sam)

# ======== Post-process mask (LCC) ========
def largest_connected_component(mask):
    mask = mask.astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    
    if num_labels <= 1:  # No foreground objects
        return np.zeros_like(mask, dtype=bool)
    
    largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    return (labels == largest_label)

# ======== Segment and visualize (safe) ========
def segment_geotiff(image_path, bbox, output_dir, predictor):
    with rasterio.open(image_path) as src:
        img_array = src.read([1, 2, 3])  # [C, H, W]
        img = np.moveaxis(img_array, 0, -1)  # [H, W, C]
        original_shape = img.shape[:2]  # (H, W)
        img = resize(img, (1024, 1024), preserve_range=True).astype(np.uint8)  # Downsample

    # Scale bbox to match resized image
    y_scale = 1024 / original_shape[0]
    x_scale = 1024 / original_shape[1]
    scaled_bbox = [
        bbox[0] * x_scale, bbox[1] * y_scale,
        bbox[2] * x_scale, bbox[3] * y_scale
    ]

    predictor.set_image(img)

    masks, scores, _ = predictor.predict(
        box=np.array([scaled_bbox]),
        multimask_output=False
    )

    best_mask = masks[0]
    
    if best_mask.sum() == 0:
        print(f"[WARNING] Empty mask for {os.path.basename(image_path)}, bbox={bbox}")
        return

    refined_mask = largest_connected_component(best_mask)

    if refined_mask.sum() == 0:
        print(f"[WARNING] No connected component found for {os.path.basename(image_path)}")
        return

    overlay = img.copy()
    overlay[refined_mask] = (overlay[refined_mask] * 0.5 + np.array([255, 0, 0]) * 0.5).astype(np.uint8)

    blended_image = Image.fromarray(overlay)
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, os.path.basename(image_path).replace('.tif', '_segmented.jpg'))
    blended_image.save(output_path)

    print(f"Saved: {output_path}")
    torch.cuda.empty_cache()


# ======== Batch mode with CSV ========
def batch_segment_from_csv(image_dir, csv_file, output_dir, predictor):
    df = pd.read_csv(csv_file)
    tifs = glob.glob(os.path.join(image_dir, '*.tif'))

    for tif in tifs:
        img_name = os.path.basename(tif)
        matches = df[df['tile_name'].str.startswith(os.path.splitext(img_name)[0])]

        if matches.empty:
            print(f"[INFO] No bbox found for {img_name}, skipping.")
            continue

        for _, row in matches.iterrows():
            bbox = [row['bbox_x1'], row['bbox_y1'], row['bbox_x2'], row['bbox_y2']]
            segment_geotiff(tif, bbox, output_dir, predictor)




In [2]:
# ======== Run Example ========
batch_segment_from_csv(
    image_dir="C:/Users/mza0288/Documents/Code/test_sample",
    csv_file="sam_test/final_rect_highest_conf.csv",
    output_dir="segmented_geotiff_output",
    predictor=predictor
)


Saved: segmented_geotiff_output\m_3108501_ne_16_030_20211103_segmented.jpg
Saved: segmented_geotiff_output\m_3108501_ne_16_030_20211103_segmented.jpg
Saved: segmented_geotiff_output\m_3108501_ne_16_030_20211103_segmented.jpg
Saved: segmented_geotiff_output\m_3108501_ne_16_030_20211103_segmented.jpg
Saved: segmented_geotiff_output\m_3108501_ne_16_030_20211103_segmented.jpg
Saved: segmented_geotiff_output\m_3108501_ne_16_030_20211103_segmented.jpg
Saved: segmented_geotiff_output\m_3108501_ne_16_030_20211103_segmented.jpg
Saved: segmented_geotiff_output\m_3108501_ne_16_030_20211103_segmented.jpg
Saved: segmented_geotiff_output\m_3108501_ne_16_030_20211103_segmented.jpg
Saved: segmented_geotiff_output\m_3108501_ne_16_030_20211103_segmented.jpg
Saved: segmented_geotiff_output\m_3108501_ne_16_030_20211103_segmented.jpg
Saved: segmented_geotiff_output\m_3108501_ne_16_030_20211103_segmented.jpg
Saved: segmented_geotiff_output\m_3108501_ne_16_030_20211103_segmented.jpg
Saved: segmented_geotiff_

In [12]:
# !pip install git+https://github.com/facebookresearch/segment-anything.git